# 05 - Dataset corrigé, séparé du brut

Ce notebook ne remplace pas le brut. Il calcule une vue corrigée distincte : pondération par récence, pondération par taille d'échantillon et correction simple de l'effet institut.

In [ ]:
from __future__ import annotations

from datetime import date
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

try:
    from IPython.display import display
except ImportError:
    display = print

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
pio.renderers.default = "json"

from presidentielle2027.adjustments.house_effects import apply_house_effect_adjustment, estimate_house_effects
from presidentielle2027.adjustments.recency_weighting import compute_recency_weights
from presidentielle2027.adjustments.sample_size_weighting import compute_sample_size_weights
from presidentielle2027.analytics.trends import smooth_candidate_trends
from presidentielle2027.dashboard.colors import get_political_color
from presidentielle2027.extraction.canonicalization import canonicalize_candidate_fields, is_generic_bloc_label

path = REPO_ROOT / "data" / "processed" / "wikipedia_2027_polls_normalized_v2.csv"
frame = pd.read_csv(path)
canonical = frame.apply(lambda row: canonicalize_candidate_fields(row.get("candidate_name"), row.get("candidate_party"), row.get("political_family")), axis=1, result_type="expand")
canonical.columns = ["candidate_name", "candidate_party", "political_family"]
frame[["candidate_name", "candidate_party", "political_family"]] = canonical
frame["publication_date"] = pd.to_datetime(frame["publication_date"], errors="coerce")
frame["estimate_percent"] = pd.to_numeric(frame["estimate_percent"], errors="coerce")
frame["sample_size"] = pd.to_numeric(frame.get("sample_size"), errors="coerce")
frame = frame.loc[(frame["round"] == "first_round") & (~frame["candidate_name"].map(is_generic_bloc_label))].copy()
frame["recency_weight"] = compute_recency_weights(frame["publication_date"], reference_date=date.today())
frame["sample_size_weight"] = compute_sample_size_weights(frame["sample_size"])
scenario_catalog = frame.groupby("scenario_name", dropna=False).agg(rows=("poll_id", "count"), pollsters=("polling_company", "nunique"), last_publication=("publication_date", "max")).sort_values(["rows", "last_publication"], ascending=[False, False])
SCENARIO_NAME = scenario_catalog.index[0]
scenario_frame = frame.loc[frame["scenario_name"] == SCENARIO_NAME].copy()
scenario_frame = smooth_candidate_trends(scenario_frame)
effects = estimate_house_effects(scenario_frame)
scenario_frame = apply_house_effect_adjustment(scenario_frame, effects)
pd.Series({"scenario": SCENARIO_NAME, "rows": len(scenario_frame), "pollsters": scenario_frame["polling_company"].nunique(), "house_effect_rows": len(effects)})

In [ ]:
summary_rows = []
for candidate_name, group in scenario_frame.groupby("candidate_name", dropna=False):
    recency_weight_sum = group["recency_weight"].sum()
    sample_weight_sum = group["sample_size_weight"].sum()
    party = group["candidate_party"].dropna().iloc[0] if group["candidate_party"].notna().any() else None
    summary_rows.append({
        "candidate_name": candidate_name,
        "candidate_party": party,
        "raw_mean": group["estimate_percent"].mean(),
        "recency_weighted_mean": (group["estimate_percent"] * group["recency_weight"]).sum() / recency_weight_sum if recency_weight_sum > 0 else group["estimate_percent"].mean(),
        "sample_weighted_mean": (group["estimate_percent"] * group["sample_size_weight"]).sum() / sample_weight_sum if sample_weight_sum > 0 else group["estimate_percent"].mean(),
        "house_effect_adjusted_mean": group["adjusted_estimate_house_effect"].mean(),
        "polls": group["poll_id"].nunique() if "poll_id" in group.columns else len(group),
    })

summary = pd.DataFrame(summary_rows).sort_values("raw_mean", ascending=False)
display(summary)
effects.sort_values(["candidate_name", "house_effect"], ascending=[True, False]).head(40)

In [ ]:
raw_figure = go.Figure()
corrected_figure = go.Figure()
for _, row in summary.iterrows():
    color = get_political_color(row.get("candidate_party"), None)
    raw_figure.add_trace(go.Bar(x=[row["candidate_name"]], y=[row["raw_mean"]], name=row["candidate_name"], marker_color=color, showlegend=False))
    corrected_figure.add_trace(go.Bar(x=[row["candidate_name"]], y=[row["house_effect_adjusted_mean"]], name=row["candidate_name"], marker_color=color, showlegend=False))

raw_figure.update_layout(title=f"Scénario brut - {SCENARIO_NAME}", xaxis_title="Candidats", yaxis_title="Intentions de vote (%)", paper_bgcolor="white", plot_bgcolor="white")
raw_figure.update_xaxes(tickangle=-35)
raw_figure.update_yaxes(showgrid=True, gridcolor="#e5e5e5")
corrected_figure.update_layout(title=f"Scénario corrigé - {SCENARIO_NAME}", xaxis_title="Candidats", yaxis_title="Intentions de vote corrigées (%)", paper_bgcolor="white", plot_bgcolor="white")
corrected_figure.update_xaxes(tickangle=-35)
corrected_figure.update_yaxes(showgrid=True, gridcolor="#e5e5e5")
raw_figure

In [ ]:
corrected_figure